# Data Analysis · Week 10
## User-defined functions

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

You already use `AVERAGE` in a sheet without knowing how it adds or divides. Today you learn to
build your own.

The convincing argument is not "reusing code". It is that **a function can be tested on its
own**, and a formula pasted into three hundred cells cannot.

By the end of this notebook you will be able to:

1. Explain what a function solves, beyond saving lines.
2. Define a function with `def`, with a name, parameters and a body.
3. Tell a parameter from an argument.
4. Return a value with `return`, and say how that differs from printing it.
5. Recognise a name's scope and why what is inside does not come out.

### How to use this notebook

Run the cells in order. Five fail on purpose and carry a comment saying so.

The case from here to week 13 is a finance one: the monthly payment on a loan.

---
# Block 1 · Why functions exist

Not to write less. To have **one single place** where the calculation can be right or wrong.

Here is the same calculation repeated by hand, for two loans:

In [ ]:
# Loan A
i = 0.18 / 12
payment_a = 250000 * (i * (1 + i) ** 36) / ((1 + i) ** 36 - 1)

# Loan B
i = 0.24 / 12
payment_b = 120000 * (i * (1 + i) ** 24) / ((1 + i) ** 24 - 1)

print(f"A: {payment_a:,.2f}")
print(f"B: {payment_b:,.2f}")

It works, and it has two problems that do not show when you read it.

The formula is written twice, so if it is wrong every copy has to be found and fixed. And the
variable `i` was reused: the second line wrote over the first, and if somebody moves the blocks
around the result changes without warning.

The same arithmetic, packaged:

In [ ]:
def monthly_payment(principal, annual_rate, months):
    i = annual_rate / 12
    factor = (1 + i) ** months
    return principal * (i * factor) / (factor - 1)


payment_a = monthly_payment(250000, 0.18, 36)
payment_b = monthly_payment(120000, 0.24, 24)

print(f"A: {payment_a:,.2f}")
print(f"B: {payment_b:,.2f}")

The same two numbers. The difference is what happens when the formula is wrong: above, every copy
has to be fixed; below, it gets fixed once and both calls come out right.

## The real argument

A function **can be tested on its own**. A formula pasted into three hundred cells can only be
checked cell by cell, and nobody does that.

Testing means this: you give it inputs whose answer you already know and check that it returns
them.

In [ ]:
# A loan of 12,000 at 0.0000001 % a year over 12 months should pay almost 1,000 a
# month, because there is practically no interest.
print("Practically interest-free:", round(monthly_payment(12000, 0.0000001, 12), 2))

# And the payment always has to exceed principal over months, because interest exists.
print("Payment:", round(monthly_payment(250000, 0.18, 36), 2))
print("Principal over months:", round(250000 / 36, 2))
print("Is the payment larger?", monthly_payment(250000, 0.18, 36) > 250000 / 36)

Those three checks fit in one cell and can be rerun every time somebody touches the formula. That
is what the spreadsheet does not give you.

---
# Block 2 · How a function is written

Five parts, and each has a rule that is not negotiable.

| Part | What it is | In the example |
|---|---|---|
| `def` | The word that declares it | `def` |
| Name | What it gets called afterwards | `monthly_payment` |
| Parameters | The slots to be filled | `principal, annual_rate, months` |
| Body | The calculation, indented | The three lines inside |
| `return` | What it hands back when it ends | The computed payment |

In [ ]:
def monthly_payment(principal, annual_rate, months):
    """Work out the fixed monthly payment on a loan.

    principal    what is lent, in pesos
    annual_rate  the nominal annual rate, as a decimal: 0.18 is 18 %
    months       the term
    """
    i = annual_rate / 12
    factor = (1 + i) ** months

    return principal * (i * factor) / (factor - 1)


payment = monthly_payment(250000, 0.18, 36)
print(f"Monthly payment: ${payment:,.2f}")

**`def` and the name.** The name says what it returns, not what it does inside.
`monthly_payment`, not `calculate_stuff`.

**The parameters.** The three slots the function needs. Calling it fills them in that same order.

**The docstring**, that triple-quoted string. It explains what the function is for, and whoever
uses it reads that instead of the body. Python keeps it and it can be looked up.

In [ ]:
print(monthly_payment.__doc__)

In [ ]:
help(monthly_payment)

**`return`** hands the result back and ends the function right there. Whatever follows does not
run.

In [ ]:
def with_dead_code(amount):
    return amount * 1.16
    print("This line never executes")


print(with_dead_code(1000))
print("And no message from inside appeared.")

## A parameter is not an argument

They get confused constantly and the distinction is simple.

**The parameter** is the slot you leave when defining it: `principal`, `annual_rate`, `months`.

**The argument** is the value that arrives when calling it: `250000`, `0.18`, `36`.

The parameter lives in the definition and the argument in the call. One is the drawer's label,
the other is what you put in.

In [ ]:
def describe_loan(principal, annual_rate, months):
    """The parameters are principal, annual_rate and months."""
    return f"{principal:,} pesos at {annual_rate:.0%} over {months} months"


# Here the arguments are 250000, 0.18 and 36.
print(describe_loan(250000, 0.18, 36))

# And they can be passed by name, in any order.
print(describe_loan(months=24, principal=120000, annual_rate=0.24))

Passing them by name costs more letters and removes all doubt. With three bare numbers, nobody
reading `monthly_payment(250000, 0.18, 36)` can swear which is which without going to look at the
definition.

In [ ]:
# FAILS ON PURPOSE, two different ways depending on which two you swap.
import itertools

for order in itertools.permutations([250000, 0.18, 36]):
    label = f"{order[0]:>10,} {order[1]:>10,} {order[2]:>10,}"
    try:
        print(f"{label} -> {monthly_payment(*order):>18,.2f}")
    except OverflowError as e:
        print(f"{label} -> OverflowError: {e}")

Six possible orders and three outcomes.

One is right, 9,038.10. Two blow up with `OverflowError`, because raising 1.015 to the two
hundred and fifty thousandth runs past what fits in a decimal. And **three return a number
without protest.**

The worst of the three is `monthly_payment(0.18, 250000, 36)`, which gives 3,750.00. That is a
perfectly believable monthly payment for a loan, and it was computed with the principal sitting
where the rate belongs.

The ones that blow up warn you. The 3,750 goes into the report.

In [ ]:
# Which is why passing by name pays off when there are several of the same type.
print(monthly_payment(principal=250000, annual_rate=0.18, months=36))

# And that way the order you write them in stops mattering.
print(monthly_payment(months=36, principal=250000, annual_rate=0.18))

## Reused

In [ ]:
loans = [
    (250000, 0.18, 36),
    (120000, 0.24, 24),
    (80000, 0.15, 12),
]

print(f"{'Principal':>10}{'Rate':>7}{'Months':>8}{'Payment':>12}")
print("-" * 37)
for principal, rate, months in loans:
    print(f"{principal:>10,}{rate:>7.0%}{months:>8}"
          f"{monthly_payment(principal, rate, months):>12,.2f}")

Three loans, one formula. Add a fourth to the list and the loop is untouched.

## Returning is not printing

This is the error that turns up most in the first assignment with functions.

In [ ]:
# FAILS ON PURPOSE. The function prints instead of returning.
def payment_that_prints(principal, annual_rate, months):
    i = annual_rate / 12
    factor = (1 + i) ** months
    print(principal * (i * factor) / (factor - 1))


total = payment_that_prints(250000, 0.18, 36)

print("What ended up in total:", total)
print("Its type:", type(total))

The number appeared on screen and `total` came back as `None`. A function that only prints is a
dead end: it cannot be added, stored or plotted.

And the `None` does not blow up there. It blows up three lines later.

In [ ]:
# FAILS ON PURPOSE. The None from above, used as if it were a number.
try:
    print(total * 36)
except TypeError as e:
    print("TypeError:", e)

That is the pattern to recognise: **a `TypeError` mentioning `NoneType` almost always means a
function is missing its `return`.**

The rule: the function returns, and whoever calls it decides whether to print.

In [ ]:
def payment_that_returns(principal, annual_rate, months):
    i = annual_rate / 12
    factor = (1 + i) ** months
    return principal * (i * factor) / (factor - 1)


total = payment_that_returns(250000, 0.18, 36)

print("The payment:", round(total, 2))
print("Over 36 months:", round(total * 36, 2))
print("Interest:", round(total * 36 - 250000, 2))

The same calculation, and now the result is good for three more things.

## A function that uses another

With the one above, the function that reports the whole loan can be written without repeating the
formula.

In [ ]:
def loan_summary(principal, annual_rate, months):
    """Returns the monthly payment, the total paid and the interest."""
    payment = monthly_payment(principal, annual_rate, months)
    total = payment * months
    return payment, total, total - principal


payment, total, interest = loan_summary(250000, 0.18, 36)

print(f"Monthly payment: {payment:>12,.2f}")
print(f"Total paid:      {total:>12,.2f}")
print(f"Interest:        {interest:>12,.2f}")
print(f"Interest is {interest / 250000:.1%} of the principal")

`return payment, total, total - principal` hands back three things at once, packed into a tuple.
The line receiving it unpacks them into three names.

And notice what is **not** there: `loan_summary` does not rewrite the payment formula. It asks for
it. If tomorrow you find the formula was wrong, you fix it in one place and both functions come
out right.

---
# Block 3 · Where each name lives

What gets declared inside a function exists only while that function runs.

In [ ]:
def compute(principal):
    fee = principal * 0.02
    return principal + fee


print(compute(250000))

# FAILS ON PURPOSE. fee was born and died inside the function.
try:
    print(fee)
except NameError as e:
    print("NameError:", e)

Variables born inside a function belong to it. They are created when you call it and disappear
when it ends.

That is not a limitation, it is the guarantee that a function cannot break the rest of the
program by accident. And it brings a good consequence: **you can reuse the same name without
worry.**

In [ ]:
def one(x):
    factor = x * 2
    return factor


def another(x):
    factor = x * 100      # the same name, and they do not collide
    return factor


print(one(5), another(5))

One function's `factor` and the other's are two different variables that will never meet.

## What can be seen from inside

A function can **read** a name from outside. That works and is nearly always a bad idea.

In [ ]:
TAX = 0.16                    # a constant of the program

def with_tax(amount):
    return amount * (1 + TAX)  # reads TAX from outside


print(with_tax(1000))

It works because `TAX` is defined when the function runs. The problem shows up when the function
moves to another file: it takes its body and leaves `TAX` behind.

In [ ]:
# FAILS ON PURPOSE. The same function, without the constant it took for granted.
def with_duty(amount):
    return amount * (1 + RATE_THAT_DOES_NOT_EXIST)


try:
    print(with_duty(1000))
except NameError as e:
    print("NameError:", e)

The version that survives the move receives everything it needs.

In [ ]:
def with_tax_portable(amount, tax=0.16):
    """Everything it needs comes in through the door."""
    return amount * (1 + tax)


print(with_tax_portable(1000))
print(with_tax_portable(1000, 0.08))      # border zone

That `tax=0.16` is a default argument, and it is next week's topic. It appears here so you can see
the solution exists.

## Modifying from inside

Reading from outside works. Assigning does not.

In [ ]:
counter = 0

def add_one():
    counter = 1     # this creates a NEW, local variable
    return counter


print("Returns:", add_one())
print("And the outer one is still:", counter)

The inner one and the outer one share a name and are not the same. The assignment created a local
that died when the function ended.

There is a word to force the opposite, `global`, and this course does not use it. A function that
modifies outside variables is exactly the one that cannot be tested on its own, which is all we
are trying to avoid.

## Four errors on your first function

**Forgetting the `return`.** The function runs, computes correctly and returns `None`. The error
appears lines later.

**Predict before you run.** What does this program print?

- **A.** 42, because it multiplies by two.
- **B.** `None`, because the function is missing its `return`.
- **C.** 21, because `n` did not change.
- **D.** An error, because the function does nothing.

In [ ]:
def double(n):
    n * 2


result = double(21)
print(result)

The answer is **B**. The multiplication happened, the result was computed, and nobody returned it.
Python hands back `None` when a function ends without a `return`.

**Printing instead of returning.** You saw it.

**Calling it before defining it.** Python reads top to bottom.

In [ ]:
# FAILS ON PURPOSE. The call comes before the definition.
try:
    print(not_yet_defined(10))
except NameError as e:
    print("NameError:", e)


def not_yet_defined(x):
    return x * 2

In a notebook this bites differently: if you run the definition cell and **then** one further up,
the function does exist, because the state is whatever the last cell you executed left behind, not
the order on screen.

It is the same warning from week 3, and this is where it starts costing.

**Relying on outside variables.** You saw it with `TAX`.

---
# Exercises

The solutions sit at the very bottom of the notebook.

## Writing functions

### Exercise 1 · Three short functions

Write three functions with docstrings, each with two parameters and a `return`:

1. `share(part, total)` returning what fraction of the total the part is.
2. `change(current, previous)` returning the percentage change.
3. `discounted(price, discount)` returning the final price.

Test each with two cases.

### Exercise 2 · Each one's edge case

For all three, find the input value that breaks them and prove it.

Hint: think about a total of zero, a previous of zero, and a discount of 120 %.

### Exercise 3 · Returning several things

Write `statistics_of(numbers)` taking a list and returning four values: the sum, the average, the
largest and the smallest. Unpack them into four names when calling it.

### Exercise 4 · One that uses another

Write `amortisation(principal, annual_rate, months)` that uses `monthly_payment` and returns a
list of tuples, one per month, with the month number, that month's interest, the principal repaid
and the remaining balance.

Each month's interest is the balance times the monthly rate. The principal repaid is the payment
minus the interest.

Check that the last month's balance ends up practically at zero.

## Scope and errors

### Exercise 5 · The `None` that blows up later

Deliberately write a function with no `return`, store it in a variable, then provoke the three
different errors that `None` can cause: adding it, indexing it and calling a method on it.

Write down each message.

### Exercise 6 · Portable or not

This function depends on something it does not receive:

```python
FEE = 0.02

def total_with_fee(amount):
    return amount * (1 + FEE)
```

Rewrite it to be portable, and show that the first breaks when the constant is deleted and the
second does not.

### Exercise 7 · The repeated name

Write two functions that internally use a variable called `total` for different things, plus a
`total` outside both. Print all three and check that none interferes with the others.

## With your own field

### Exercise 8 · Package a calculation of your own

Write a function that solves a real calculation from your field, with at least two parameters and
a `return`. Write its docstring and test it with three cases, one of them at the boundary.

The function may not print anything. It only receives and returns.

The test: delete a line from the body. If all three tests still pass, your cases were not testing
anything.

---
## Three ideas to take away

**A function can be tested on its own.** That is the real argument. Saving lines is only the most
visible side effect.

**Returning is not printing.** A function that only prints hands back `None`, and that `None`
blows up three lines later with a `TypeError` that never mentions the function.

**What is inside stays inside.** Which is why you can reuse the same names in two functions
without collision, and why a well-written function cannot break the rest of the program.

Next session is default arguments, built-in functions and the modules that already ship with
Python.

---
# Solutions

### Exercise 1

```python
def share(part, total):
    """What fraction of the total the part represents, as a decimal."""
    return part / total


def change(current, previous):
    """The percentage change against the previous period, as a decimal."""
    return (current - previous) / previous


def discounted(price, discount):
    """The final price after applying a discount given as a decimal."""
    return price * (1 - discount)


print(f"{share(5074, 148230):.2%}")
print(f"{change(148230, 96400):+.1%}")
print(f"{discounted(8990, 0.15):,.2f}")
```

Note the `+` in `{:+.1%}`: it forces the sign, so growth reads `+53.8%` and a fall reads `-12.0%`.
In a variance report that is worth more than the number alone.

### Exercise 2

```python
for f, args, label in [(share, (5074, 0), "total of zero"),
                       (change, (100, 0), "previous of zero"),
                       (discounted, (8990, 1.2), "discount of 120 %")]:
    try:
        print(f"{label:<20} -> {f(*args)}")
    except ZeroDivisionError as e:
        print(f"{label:<20} -> ZeroDivisionError: {e}")
```

The first two blow up and the third does not: it returns a negative price, `-1798.0`.

That is the dangerous one. An error that blows up warns you; one that returns a negative price
goes onto the invoice. If the function is going to be used for real, that is where a validation
belongs.

### Exercise 3

```python
def statistics_of(numbers):
    """Returns sum, average, largest and smallest of a list of numbers."""
    return sum(numbers), sum(numbers) / len(numbers), max(numbers), min(numbers)


total, average, largest, smallest = statistics_of([23200, 42800, 82700, 24500, 24500])

print(f"Sum:      {total:>10,}")
print(f"Average:  {average:>10,.2f}")
print(f"Largest:  {largest:>10,}")
print(f"Smallest: {smallest:>10,}")
```

With an empty list it blows up on the division. Worth deciding whether that is right: for a
statistics function it probably is, because the average of nothing does not exist.

### Exercise 4

```python
def amortisation(principal, annual_rate, months):
    """One tuple per month: number, interest, principal repaid and balance."""
    payment = monthly_payment(principal, annual_rate, months)
    i = annual_rate / 12
    balance = principal
    rows = []
    for month in range(1, months + 1):
        interest = balance * i
        repaid = payment - interest
        balance -= repaid
        rows.append((month, interest, repaid, balance))
    return rows


table = amortisation(250000, 0.18, 36)

print(f"{'Month':>6}{'Interest':>12}{'Repaid':>12}{'Balance':>14}")
for month, interest, repaid, balance in table[:3]:
    print(f"{month:>6}{interest:>12,.2f}{repaid:>12,.2f}{balance:>14,.2f}")
print("  ...")
for month, interest, repaid, balance in table[-2:]:
    print(f"{month:>6}{interest:>12,.2f}{repaid:>12,.2f}{balance:>14,.2f}")

print(f"\nFinal balance: {table[-1][3]:.10f}")
```

The final balance lands on something like `0.0000000005`, not exactly zero. It is the same binary
rounding from week 4: every month drags a fraction of a cent along.

In a real system the last payment is adjusted to close exactly, and that is a business decision,
not a flaw in the formula.

### Exercise 5

```python
def no_return(x):
    x * 2


empty = no_return(21)

for label, action in [("adding it", lambda: empty + 1),
                      ("indexing it", lambda: empty[0]),
                      ("calling a method", lambda: empty.upper())]:
    try:
        action()
    except TypeError as e:
        print(f"{label:<18} TypeError: {e}")
    except AttributeError as e:
        print(f"{label:<18} AttributeError: {e}")
```

All three mention `NoneType` and none mentions `no_return`. That is why recognising the word
`NoneType` in a message is worth so much: it is the clue that the problem sits in a function that
did not return, not in the line that blew up.

### Exercise 6

```python
def total_with_fee_portable(amount, fee=0.02):
    """Everything it needs arrives as a parameter."""
    return amount * (1 + fee)


print(total_with_fee_portable(1000))
print(total_with_fee_portable(1000, 0.05))

def total_with_fee(amount):
    return amount * (1 + FEE_I_NEVER_DEFINED)

try:
    total_with_fee(1000)
except NameError as e:
    print("The first version:", e)
```

The portable one works on its own and gained flexibility besides: the fee can now be changed per
call without touching the function. That is the pattern, and week 11 formalises it.

### Exercise 7

```python
total = "the outer one"

def sum_prices(prices):
    total = sum(prices)
    return total

def count_items(items):
    total = len(items)
    return total

print(sum_prices([100, 200, 300]))
print(count_items(["a", "b", "c", "d"]))
print(total)
```

You get 600, 4 and `the outer one`. Three variables sharing a name and none of them knows about
the others.

That the outer one is text and the inner ones are numbers is deliberate: if they collided,
something would have blown up.

### Exercise 8

There is no published solution, because the calculation differs for everyone. It is graded on four
things: that it has a docstring, that it prints nothing, that the three tests include an edge
case, and that deleting a line from the body makes at least one test fail.

That last one is what really measures. A test that still passes with the function broken was not
testing anything.